# 27 · Rate ratio with an interval, and the Two-by-two table, checked against statsmodels

## What this checks

A **rate ratio** says how many times higher one group's rate is than another's. A **95% confidence interval** gives
the range of ratios the data are consistent with.

Fieldwork says the cholera death rate in houses supplied by the Southwark and Vauxhall Company was **8.40 times** the
rate in houses supplied by Lambeth. Its 95% interval is **6.84 to 10.32**. Does an independent implementation get the
same three numbers?

Two widgets are checked:

- **Rate ratio with an interval**, release 0.1.0. It computed these figures first. It is now retired.
- **Two-by-two table**, release 0.8.3. It took over this job on 5 November 2026. It reads a table as events and units
  at risk.

This is the claim the South London example exists to make. An epidemiologist will check it against their own
calculation before trusting anything else on the screen. Until this check, it rested on Fieldwork's own tests alone.

## How it is checked

We use **statsmodels**, a Python statistics library by other authors. It has the same method Fieldwork declares,
written independently.

**The method matters.** A rate ratio is not a risk ratio. A risk ratio compares the share of people affected, not
the rate. The two have different intervals:

- **Fieldwork's rate interval** is `ln(RR) ± z·sqrt(1/a + 1/b)`, over the event counts.
- **A risk ratio's interval**, over the same table, would use `sqrt(1/a - 1/n1 + 1/b - 1/n2)`, and give different
  bounds.

Comparing against the wrong one would have produced a confident disagreement about nothing. So we compare against
statsmodels' `wald-log` method, which is the one Fieldwork declares.

We also show a second method, `score`, beside it. It estimates the spread differently. The gap between the two shows
what Fieldwork's choice of method costs. We do not require them to agree.

The data are John Snow's 1855 figures, compared four ways:

- deaths per house, by water company;
- deaths per person, by the supply groups his table declares;
- the intermingled area against Lambeth only;
- Southwark and Vauxhall only against the intermingled area.

## What should happen

1. **The ratio** within 0.000000001 of statsmodels. It is one division, with nothing to approximate.
2. **Both interval bounds** within 0.000001 of statsmodels' `wald-log` method.
3. **The z value.** Fieldwork hard-codes z, the number of standard errors for 95%, as 1.9600. It must be within
   0.00005 of the exact value from scipy.
4. **No multiplier in the ratio.** A rate can be "per 10,000", but the ratio must not carry that. We recompute the
   ratio from raw counts, so a multiplier that leaked in would show.
5. **The score interval** is shown beside the declared one, never required to match.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["statsmodels", "scipy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_27_rate_ratio import run

report = run(Path.cwd() / "fixtures" / "rate-ratio-snow-001.json")
print("agrees:", report["agrees"])
for c in report["comparisons"]:
    f, s = c["fieldwork"], c["scoreMethod"]
    print(f"{c['note']}: {f['ratio']:.4f} [{f['lower']:.4f}, {f['upper']:.4f}] "
          f"· score [{s['lower']:.4f}, {s['upper']:.4f}]")


## The result on record

For Rate ratio with an interval, recorded on 26 October 2026, against release 0.1.0: **Agrees**. On 26 November 2026
it was run again on the same data, not on new data, because the widget is retired and nothing can export its data
now. It still agrees.

All four comparisons agree in full:

- **Per house:** 8.4019, with a 95% interval of 6.8410 to 10.3189. The score method gives 6.8434 to 10.3152.
- **Per person, by supply group:** 5.3511, from 3.3549 to 8.5349.
- **The intermingled area against Lambeth only:** 2.3090, from 1.4456 to 3.6881.
- **Southwark and Vauxhall only against the intermingled area:** 2.3175, from 2.0924 to 2.5668.

Ratios agree within 0.000000001, and both bounds within 0.000001.

Running the cell above repeats the check on the same data.

## Since 5 November 2026: the same comparisons, from the Two-by-two table

On 5 November 2026 the Two-by-two table took over from Rate ratio with an interval. The cell below runs the same four
comparisons on data made by the Two-by-two table. Its tables come from the South London workflows:

- Snow's Table IX, as the example carries it;
- the supply-group totals, which the example computes from Tables VIII and XII.

The criteria and the method are the same. The cell above keeps the retired widget's data, as the record of what it
was checked against.

For the Two-by-two table, first recorded on 5 November 2026, against release 0.7.0. Rerun on 26 November 2026,
against release 0.8.3: **Agrees**. Every ratio and every bound equals the retired widget's to the last digit.

In [ ]:
two_by_two = run(Path.cwd() / "fixtures" / "two-by-two-rates-snow-001.json")
print("fixture:", two_by_two["fixture"]["operation"]["widget"], two_by_two["fixture"]["operation"]["release"])
for c in two_by_two["comparisons"]:
    f = c["fieldwork"]
    print(f"{'OK ' if c['agrees'] else 'DIFF'} {f['ratio']:.4f} [{f['lower']:.4f}, {f['upper']:.4f}]  {c['note']}")
print("AGREES" if two_by_two["agrees"] else "DIVERGES")

## What this check found

On its first run, every interval missed by more than 0.000001, while every ratio agreed exactly. That pattern points
to a rounded constant, not a wrong formula.

Fieldwork had hard-coded the 95% value of z as **1.9600**. The exact value is **1.959963984540054**. That moved Snow's
per-house lower bound from 6.8409 to 6.8410.

At two decimals it makes no difference. But it is exactly what an independent check exists to find before a reader
does. It was fixed, and the data were exported again. The cell below is the comparison that caught it.

In [ ]:
from scipy.stats import norm
print("scipy  :", norm.ppf(0.975))
print("used   :", report["zQuantile"]["fieldwork"], "— agrees:", report["zQuantile"]["agrees"])


## What this does not show

- **That the comparison is fair.** The two companies' customers were not randomly assigned. No interval speaks to
  confounding, that is, to other differences between the groups. Snow's own argument rests on the two supplies being
  mixed within the same streets. That is a fact about the setting, not about the data.
- **That this interval is adequate at small counts.** The Lambeth-only group has 18 deaths. An exact interval would
  be the better comparison there. statsmodels documents that method as not yet available. We record that, and do not
  work around it: an exact interval written by hand would be our own arithmetic checking our own.
- **What the right denominator is.** Houses and people give different ratios for the same outbreak, 8.40 and 5.35.
  Choosing between them is the practitioner's call.